# 04 — ECAPA-TDNN Speaker Verification

## Objective

Evaluate whether ECAPA-TDNN embeddings distinguish utterances from the same speaker from utterances spoken by different people.

The experiment builds positive and negative verification trials, extracts or loads ECAPA embeddings, computes cosine-similarity scores, and reports the Equal Error Rate (EER), its threshold, and the corresponding FAR/FRR.

**Protocol:** 100 speakers, at least 5 distinct videos per eligible speaker, 10 positive trials and 10 negative trials per selected speaker. The random seed is `42`.

> Reproducibility note: changing the trial-generation implementation can change the exact sampled pairs even when counts and labels are preserved. Compare with the previously saved trial CSV before treating results as numerically equivalent.

In [87]:
from pathlib import Path
import sys

import pandas as pd
from tqdm.auto import tqdm

# Locate the repository root from the notebook's current directory.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        parent
        for parent in (cwd, *cwd.parents)
        if (parent / "src").is_dir() and (parent / "notebooks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the repository root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

VOXCELEB_WAV_DIR = PROJECT_ROOT / "voxceleb1" / "wav"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
TRIALS_DIR = PROJECT_ROOT / "data" / "trials"
CACHE_DIR = PROJECT_ROOT / "data" / "cache"

print("Project root:", PROJECT_ROOT)
print("VoxCeleb WAV directory exists:", VOXCELEB_WAV_DIR.is_dir())
print("Metadata directory exists:", METADATA_DIR.is_dir())

Project root: /Users/juni/Documents/interview-anonymization
VoxCeleb WAV directory exists: True
Metadata directory exists: True


In [88]:
from src.verification_trials import resolve_audio_path

metadata_path = METADATA_DIR / "voxceleb_utterances.parquet"
utterance_df = pd.read_parquet(metadata_path)

required_columns = {"speaker_id", "video_id", "path"}
missing_columns = required_columns - set(utterance_df.columns)
if missing_columns:
    raise ValueError(f"Missing metadata columns: {sorted(missing_columns)}")

utterance_df = utterance_df.copy()
utterance_df["path"] = [
    str(resolve_audio_path(path, VOXCELEB_WAV_DIR))
    for path in tqdm(utterance_df["path"], desc="Resolving audio paths")
]

missing_paths = ~utterance_df["path"].map(lambda p: Path(p).is_file())
if missing_paths.any():
    raise FileNotFoundError(
        f"{missing_paths.sum()} audio files are missing. "
        "Check VOXCELEB_WAV_DIR and the metadata paths."
    )

print("Utterances:", len(utterance_df))
print("Speakers:", utterance_df["speaker_id"].nunique())
print("All audio paths exist.")
utterance_df.head()

Resolving audio paths: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 124843/124843 [00:06<00:00, 19513.40it/s]


Utterances: 124843
Speakers: 1166
All audio paths exist.


,speaker_id,video_id,utterance_id,path,split,language,language_confidence
0,id10001,1zcIwhmdeo4,00001,/Users/juni/Documents/interview-anonymization/...,train,english,0.99
1,id10001,1zcIwhmdeo4,00003,/Users/juni/Documents/interview-anonymization/...,train,english,0.99
2,id10001,1zcIwhmdeo4,00002,/Users/juni/Documents/interview-anonymization/...,train,english,0.99
3,id10001,7gWzIy6yIIk,00004,/Users/juni/Documents/interview-anonymization/...,train,english,0.98
4,id10001,7gWzIy6yIIk,00001,/Users/juni/Documents/interview-anonymization/...,train,english,0.98


In [89]:
from src.verification_trials import (
    select_eligible_speakers,
    build_verification_trials,
)

SEED = 42
MIN_VIDEOS_PER_SPEAKER = 5
N_SPEAKERS = 100
POS_PER_SPEAKER = 10
NEG_PER_SPEAKER = 10

eligible_speakers = select_eligible_speakers(
    utterance_df,
    min_videos_per_speaker=MIN_VIDEOS_PER_SPEAKER,
)

verification_trials, selected_speakers = build_verification_trials(
    utterances=utterance_df,
    eligible_speakers=eligible_speakers,
    seed=SEED,
    n_speakers=N_SPEAKERS,
    positive_trials_per_speaker=POS_PER_SPEAKER,
    negative_trials_per_speaker=NEG_PER_SPEAKER,
)

positive = verification_trials.loc[verification_trials["label"] == 1]
negative = verification_trials.loc[verification_trials["label"] == 0]

assert len(selected_speakers) == N_SPEAKERS
assert len(positive) == N_SPEAKERS * POS_PER_SPEAKER
assert len(negative) == N_SPEAKERS * NEG_PER_SPEAKER
assert (positive["speaker1"] == positive["speaker2"]).all()
assert (negative["speaker1"] != negative["speaker2"]).all()
assert not verification_trials[["path1", "path2"]].isna().any().any()

print("Eligible speakers:", len(eligible_speakers))
print("Selected speakers:", len(selected_speakers))
print("Positive trials:", len(positive))
print("Negative trials:", len(negative))
print("Total trials:", len(verification_trials))
verification_trials.head()

Eligible speakers: 1132
Selected speakers: 100
Positive trials: 1000
Negative trials: 1000
Total trials: 2000


,path1,path2,speaker1,speaker2,label
0,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,id10923,id10674,0
1,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,id10712,id10712,1
2,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,id11076,id11076,1
3,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,id11127,id11127,1
4,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,id10264,id10864,0


In [90]:
TRIALS_DIR.mkdir(parents=True, exist_ok=True)
trials_path = TRIALS_DIR / "voxceleb_verification_trials_legacy_reproduction.csv"

verification_trials.to_csv(trials_path, index=False)
print("Saved trials to:", trials_path)

Saved trials to: /Users/juni/Documents/interview-anonymization/data/trials/voxceleb_verification_trials_legacy_reproduction.csv


In [91]:
unique_paths = sorted(
    set(verification_trials["path1"])
    | set(verification_trials["path2"])
)

print("Trials:", len(verification_trials))
print("Unique utterances:", len(unique_paths))

Trials: 2000
Unique utterances: 1369


In [92]:
from src.ecapa import load_ecapa

classifier = load_ecapa()

In [93]:
from src.ecapa import extract_embedding
from src.embedding_cache import SQLiteEmbeddingCache

MODEL_ID = "speechbrain/spkrec-ecapa-voxceleb:extract-v1"
cache = SQLiteEmbeddingCache(
    CACHE_DIR / "ecapa_embeddings.sqlite",
    model_id=MODEL_ID,
)

embedding_cache = {}
computed_now = 0
loaded_from_cache = 0

for audio_path in tqdm(unique_paths, desc="ECAPA embeddings"):
    embedding = cache.get(audio_path)
    if embedding is None:
        embedding = extract_embedding(audio_path, classifier)
        cache.set(audio_path, embedding)
        computed_now += 1
    else:
        loaded_from_cache += 1

    embedding_cache[audio_path] = embedding

print("Embeddings in memory:", len(embedding_cache))
print("Loaded from SQLite:", loaded_from_cache)
print("Computed in this run:", computed_now)
print("Cache entries for this model:", len(cache))

ECAPA embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1369/1369 [00:00<00:00, 8546.74it/s]

Embeddings in memory: 1369
Loaded from SQLite: 1369
Computed in this run: 0
Cache entries for this model: 4419


In [94]:
from src.speaker_verification import cosine_similarity

scores = []
for row in tqdm(
    verification_trials.itertuples(index=False),
    total=len(verification_trials),
    desc="Scoring trials",
):
    scores.append(
        cosine_similarity(
            embedding_cache[row.path1],
            embedding_cache[row.path2],
        )
    )

verification_trials = verification_trials.copy()
verification_trials["score"] = scores

verification_trials[
    ["label", "speaker1", "speaker2", "score"]
].head(10)

Scoring trials: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:00<00:00, 72444.86it/s]


,label,speaker1,speaker2,score
0,0,id10923,id10674,-0.002184
1,1,id10712,id10712,0.580658
2,1,id11076,id11076,0.745721
3,1,id11127,id11127,0.650018
4,0,id10264,id10864,-0.079055
5,0,id10979,id10264,0.061914
6,1,id10153,id10153,0.702471
7,0,id10356,id10838,-0.063079
8,0,id11110,id10824,0.081987
9,1,id10590,id10590,0.310468


In [95]:
verification_trials.groupby("label")["score"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,1000.0,0.007580,0.084429,-0.283528,-0.045626,0.006835,0.058050,0.358422
1,1000.0,0.595812,0.125841,0.041993,0.514424,0.603865,0.681596,0.990775


In [96]:
from src.speaker_verification import compute_eer

labels = verification_trials["label"].to_numpy()
scores = verification_trials["score"].to_numpy()
eer_results = compute_eer(labels, scores)

print(f"EER: {eer_results['eer'] * 100:.2f}%")
print(f"EER threshold: {eer_results['threshold']:.4f}")
print(f"FAR at EER: {eer_results['far'] * 100:.2f}%")
print(f"FRR at EER: {eer_results['frr'] * 100:.2f}%")

EER: 0.55%
EER threshold: 0.2508
FAR at EER: 0.40%
FRR at EER: 0.70%


## Interpretation

A lower EER indicates better speaker-verification performance on this trial set. These results quantify how discriminative ECAPA embeddings are for this protocol; they do not by themselves establish the privacy of an anonymization method.

Record the trial-generation configuration, model version, and EER when comparing experiments. The SQLite cache avoids recomputing embeddings for unchanged audio files under the same `MODEL_ID`.